# Paper PoRT Recreated Classifier Diagnostics

This notebook diagnoses the recreated post-judgment classifier after notebook 17 showed weak held-out performance and `rethink_rate=1.0` across the smoke matrix.

It does not run the full PoRT pipeline. It focuses on label quality, split leakage, feature ablations, calibration, and classifier baselines before deciding whether a full recreated PoRT run is defensible.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'sklearn': 'scikit-learn',
}

if os.environ.get('PORT_CLASSIFIER_DIAG_TRANSFORMER_SMOKE', '').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}:
    required_packages.update({
        'torch': 'torch',
        'transformers': 'transformers>=4.38.0',
        'accelerate': 'accelerate>=0.26.0',
    })

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


## Runtime Config

Default mode is CPU-friendly and does not fine-tune a transformer. It first tries to use existing recreated classifier data from `PORT_RECREATED_ARTIFACT_DIR`, `PORT_RECREATED_ARTIFACT_ZIP_URL`, `PORT_RECREATED_ARTIFACT_ZIP_PATH`, `/kaggle/working`, or `/kaggle/input`; if none is found, it rebuilds a larger weak proxy dataset directly from public WMDP data.

Useful overrides:

- `PORT_CLASSIFIER_DIAG_SAMPLES_PER_DOMAIN=256`
- `PORT_CLASSIFIER_DIAG_WRONG_ANSWERS_PER_QUESTION=3`
- `PORT_CLASSIFIER_DIAG_TRANSFORMER_SMOKE=false`
- `PORT_RECREATED_ARTIFACT_ZIP_URL=https://.../paper_port_recreated_artifacts_bootstrap.zip`


In [ ]:
import importlib.util

runner_path = PROJECT_ROOT / 'notebooks' / 'common' / 'port_classifier_diagnostics.py'
if not runner_path.exists():
    raise FileNotFoundError(runner_path)

spec = importlib.util.spec_from_file_location('port_classifier_diagnostics', runner_path)
port_classifier_diagnostics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(port_classifier_diagnostics)

result = port_classifier_diagnostics.run(
    project_root=PROJECT_ROOT,
    is_kaggle=IS_KAGGLE,
    commit_sha=commit_sha,
)
print(json.dumps(result, indent=2, default=str)[:8000])
